# HeartGuard AI — Exploratory Data Analysis

**Week 2 | Pipeline A: UCI Heart Disease Datasets**

This notebook performs interactive EDA across all four UCI Heart Disease sites (Cleveland, Hungary, Switzerland, VA Long Beach) and the Statlog dataset.

For full automated reports see:
- `reports/EXPLORATORY_DATA_ANALYSIS_REPORT.md`
- `reports/MISSINGNESS_REPORT.md`
- `reports/DATASET_COMPATIBILITY_REPORT.md`
- `reports/STATLOG_DUPLICATE_REPORT.md`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

BASE_DIR = Path().resolve().parent
RAW_DIR = BASE_DIR / 'data' / 'raw'

UCI_COLS = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num'
]

def load_uci(name):
    path = RAW_DIR / 'uci' / f'{name}.data'
    df = pd.read_csv(path, header=None, names=UCI_COLS, na_values='?')
    df['site'] = name
    df['binary_target'] = (df['num'] >= 1).astype(float)
    df.loc[df['num'].isna(), 'binary_target'] = np.nan
    return df

datasets = {
    'cleveland':   load_uci('cleveland'),
    'hungary':     load_uci('hungary'),
    'switzerland': load_uci('switzerland'),
    'va':          load_uci('va'),
}

print('Dataset shapes:')
for name, df in datasets.items():
    print(f'  {name}: {df.shape}')

## 1. Dataset Dimensions and Data Types

In [ ]:
for name, df in datasets.items():
    print(f'\n--- {name.upper()} ---')
    print(df.dtypes)
    print(df.head(3))

## 2. Missing Values per Site

In [ ]:
for name, df in datasets.items():
    miss = df[UCI_COLS[:-1]].isna().sum()  # exclude 'num'
    miss_pct = 100 * miss / len(df)
    miss_df = pd.DataFrame({'Missing': miss, 'Missing %': miss_pct.round(1)})
    miss_df = miss_df[miss_df['Missing'] > 0]
    print(f'\n--- {name.upper()} missing values ---')
    print(miss_df if not miss_df.empty else '  None')

## 3. Target Distribution by Site

In [ ]:
for name, df in datasets.items():
    vc = df['binary_target'].value_counts(dropna=False).sort_index()
    pct = (100 * vc / len(df)).round(1)
    print(f'\n{name}: {dict(zip(vc.index, zip(vc.values, pct.values)))}')

## 4. Numeric Feature Distributions

In [ ]:
NUMERIC_FEATURES = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

for name, df in datasets.items():
    print(f'\n--- {name.upper()} numeric summary ---')
    print(df[NUMERIC_FEATURES].describe().round(2))

## 5. Zero-coded Values

In [ ]:
for name, df in datasets.items():
    print(f'\n--- {name.upper()} zero-coded counts ---')
    for col in NUMERIC_FEATURES:
        n_zero = (df[col] == 0).sum()
        if n_zero > 0:
            print(f'  {col}: {n_zero} zeros ({100*n_zero/len(df):.1f}%)')

## 6. Correlation Matrix (Cleveland)

In [ ]:
cle = datasets['cleveland']
corr_cols = NUMERIC_FEATURES + ['binary_target']
print(cle[corr_cols].corr().round(2))

## 7. Site-wise Comparison Table

In [ ]:
rows = []
for name, df in datasets.items():
    n = len(df)
    age_mean = df['age'].mean()
    age_std  = df['age'].std()
    pct_male = 100 * (df['sex'] == 1).sum() / n
    prevalence = 100 * (df['binary_target'] == 1).sum() / n
    chol_zero = (df['chol'] == 0).sum()
    rows.append({'Site': name, 'N': n,
                 'Age (mean+/-SD)': f'{age_mean:.1f}+/-{age_std:.1f}',
                 '% Male': f'{pct_male:.1f}%',
                 'Disease %': f'{prevalence:.1f}%',
                 'chol zeros': chol_zero})

pd.DataFrame(rows).set_index('Site')

## 8. Statlog Duplicate Verification (Summary)

See `reports/STATLOG_DUPLICATE_REPORT.md` for the full programmatic report.

**Key Finding (Week 2):**
- 270 of 303 Cleveland rows match exactly to all 270 Statlog rows (string-key comparison on 13 features).
- All 270 Statlog rows have exact Cleveland counterparts.
- 0 near-matches (distance <= 1.0) beyond exact matches.
- 33 unmatched Cleveland rows contain missing values (`?`) which prevent exact key matching.
- **Status:** Pending clinical feature-coding review. Not formally confirmed as exact subset.